# Chapter 4 Project 3 - Learning-Enhanced Prediction

This lab keeps learning small: generate mismatch data, fit a least-squares residual model, and compare one-step prediction errors. The learned residual improves prediction; it does not replace constraints or solver status checks in MPC.

In [ ]:
import os
from pathlib import Path

import numpy as np

from systems.mobile_robot import tracking_error_matrices
from systems.residual_models import fit_residual_least_squares, predict_residual, residual_features
from scenarios.ch4_project3_learning import generate_data, true_transition, run_project3

cwd = Path.cwd()
repo_root = cwd if (cwd / "scenarios").exists() else cwd.parent
output_root = Path(os.environ.get("THIMPC_OUTPUT_ROOT", repo_root / "outputs"))

A, B = tracking_error_matrices(dt=0.1, v_r=0.8, omega_r=0.2)
print("nominal prediction A =")
print(A)
print("nominal prediction B =")
print(B)


In [ ]:
samples = int(os.environ.get("THIMPC_CH4_PROJECT3_SAMPLES", "300"))
ridge = 1e-8

# TODO: implement or tune this design choice.
# The surrounding setup is provided so you can focus on the control idea.

errors, inputs = generate_data(samples)
measured_next = true_transition(errors, inputs, A, B)
nominal_next = errors @ A.T + inputs.reshape(-1, 1) @ B.T
residual = measured_next - nominal_next
W = fit_residual_least_squares(errors, inputs, residual, ridge=ridge)
print("first feature vector:", residual_features(errors[0], inputs[0]))
print("residual weight matrix shape:", W.shape)


## One-Step Prediction Comparison

The learned model predicts the residual of the nominal model. It improves the one-step prediction error, but constraints still have to be modeled and enforced explicitly by MPC.

In [ ]:
learned_next = nominal_next + predict_residual(W, errors, inputs)
nominal_rmse = np.sqrt(np.mean((measured_next - nominal_next) ** 2, axis=0))
learned_rmse = np.sqrt(np.mean((measured_next - learned_next) ** 2, axis=0))
print("nominal RMSE [xi, eta, psi]:", nominal_rmse)
print("learned RMSE [xi, eta, psi]:", learned_rmse)
print("learning improves prediction:", bool(np.linalg.norm(learned_rmse) < np.linalg.norm(nominal_rmse)))
print("constraints remain explicit MPC design choices; they are not learned away")


## Run the Full Application Study

In [ ]:
metrics = run_project3(samples=samples, output_dir=output_root / "ch4_project3")
metrics
